In [1]:
import pandas as pd

df = pd.read_parquet(
    r'C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_featured.parquet'
)

# Make sure date is datetime again
df['date'] = pd.to_datetime(df['date'])

print(df.shape)
print(df.columns)

(66617, 63)
Index(['date', 'station_name', 'pm25', 'latitude', 'longitude', 'n_pixels',
       'AOD_mean', 'AOD_max', 'AOD_p75', 'AOD_count', 'AOD_coverage',
       'PBLH_mean', 'PBLH_min', 'PBLH_max', 'TLML_mean', 'TLML_max',
       'moisture_mean', 'moisture_max', 'SPEED_mean', 'SPEED_min',
       'PRECTOT_sum', 'vent_coeff_mean', 'vent_coeff_min',
       'inversion_proxy_mean', 'inversion_proxy_max', 'wind_dir_mean',
       'PBLH_morning_mean', 'PBLH_morning_min', 'month', 'day_of_year',
       'day_of_week', 'year', 'is_weekend', 'season', 'month_sin', 'month_cos',
       'doy_sin', 'doy_cos', 'PM_proxy', 'PM_proxy_max', 'AOD_x_moisture',
       'AOD_x_vent', 'AOD_x_vent_min', 'AOD_x_inversion', 'AOD_x_PBLH_min',
       'trap_score', 'is_stagnant', 'rain_lag1', 'rain_lag2', 'is_rainy',
       'post_rain1', 'post_rain2', 'rain_3day', 'AOD_lag1', 'AOD_lag2',
       'AOD_roll3', 'AOD_roll7', 'station_pm_mean', 'station_pm_std',
       'station_month_pm', 'proxy_anomaly', 'lat', 'lon']

In [2]:
print(df.isna().sum().sum())   # must be 0
print(df['pm25'].describe())
print(df.dtypes.value_counts())

0
count    66617.000000
mean        52.093902
std         41.616170
min          4.880000
25%         23.850000
50%         40.080000
75%         66.794792
max        268.080000
Name: pm25, dtype: float64
float64           47
int64              8
int32              4
object             3
datetime64[ns]     1
Name: count, dtype: int64


In [12]:


import pandas as pd
import numpy as np
import os
import joblib

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics  import r2_score, mean_absolute_error
import xgboost  as xgb
import lightgbm as lgb

# ── Load final dataset ───────────────────────────────────────────────
df = pd.read_parquet(
    r'C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_featured.parquet'
)

df['date'] = pd.to_datetime(df['date'])

# ── Model save path ─────────────────────────────────────────────────
MODEL_DIR = r'C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\models'
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Columns to exclude ──────────────────────────────────────────────
EXCLUDE = {
    'pm25', 'date', 'station_name', 'station_id',
    'latitude', 'longitude',
    'aod_quality_cat',
    'station_month_aod_n',
}

feature_cols = [
    c for c in df.columns
    if c not in EXCLUDE
    and df[c].dtype in ['float64','float32','int64','int32','bool','uint8']
]

print(f"Total features : {len(feature_cols)}")

# ── Temporal split ──────────────────────────────────────────────────
SPLIT    = pd.Timestamp('2024-07-01')
train_df = df[df['date'] <  SPLIT]
test_df  = df[df['date'] >= SPLIT]

X_train, y_train = train_df[feature_cols].astype(float), train_df['pm25']
X_test,  y_test  = test_df[feature_cols].astype(float),  test_df['pm25']

print(f"Train : {X_train.shape} | Test : {X_test.shape}")

def evaluate(name, pred, y):
    r2   = r2_score(y, pred)
    mae  = mean_absolute_error(y, pred)
    rmse = np.sqrt(((pred - y)**2).mean())
    print(f"  {name:25s}  R²={r2:.4f}  MAE={mae:.2f}  RMSE={rmse:.2f}")
    return r2

preds, scores = {}, {}

# ───────────────────────────────────────────────────────────────────
# Model 1 — Random Forest
# ───────────────────────────────────────────────────────────────────
print("\n[1/4] Random Forest")
rf = RandomForestRegressor(
    n_estimators=500, max_depth=15,
    min_samples_leaf=10, max_features=0.6,
    n_jobs=-1, random_state=42
)
rf.fit(X_train, y_train)

preds['rf']  = rf.predict(X_test)
scores['rf'] = evaluate("Random Forest", preds['rf'], y_test)

# Save model
joblib.dump(rf, os.path.join(MODEL_DIR, "rf_model.pkl"))

# Feature importance
fi = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\n  Top 15 features:")
print(fi.head(15).to_string(index=False))


# ───────────────────────────────────────────────────────────────────
# Model 2 — XGBoost
# ───────────────────────────────────────────────────────────────────
print("\n[2/4] XGBoost")
xgb_m = xgb.XGBRegressor(
    n_estimators=1000, max_depth=6,
    learning_rate=0.03, subsample=0.8,
    colsample_bytree=0.7, reg_alpha=0.1,
    reg_lambda=1.0, early_stopping_rounds=50,
    random_state=42, n_jobs=-1, verbosity=0
)

xgb_m.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)

preds['xgb']  = xgb_m.predict(X_test)
scores['xgb'] = evaluate("XGBoost", preds['xgb'], y_test)

# Save model
joblib.dump(xgb_m, os.path.join(MODEL_DIR, "xgb_model.pkl"))


# ───────────────────────────────────────────────────────────────────
# Model 3 — LightGBM
# ───────────────────────────────────────────────────────────────────
print("\n[3/4] LightGBM")
lgb_m = lgb.LGBMRegressor(
    n_estimators=1000, max_depth=6,
    learning_rate=0.03, num_leaves=63,
    subsample=0.8, colsample_bytree=0.7,
    reg_alpha=0.1, reg_lambda=1.0,
    min_child_samples=20,
    random_state=42, n_jobs=-1, verbose=-1
)

lgb_m.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(100)
    ]
)

preds['lgb']  = lgb_m.predict(X_test)
scores['lgb'] = evaluate("LightGBM", preds['lgb'], y_test)

# Save model
joblib.dump(lgb_m, os.path.join(MODEL_DIR, "lgb_model.pkl"))


# ───────────────────────────────────────────────────────────────────
# Model 4 — Gradient Boosting
# ───────────────────────────────────────────────────────────────────
print("\n[4/4] Gradient Boosting")
gb = GradientBoostingRegressor(
    n_estimators=500, max_depth=5,
    learning_rate=0.05, subsample=0.8,
    min_samples_leaf=15, random_state=42
)

gb.fit(X_train, y_train)

preds['gb']  = gb.predict(X_test)
scores['gb'] = evaluate("Gradient Boosting", preds['gb'], y_test)

# Save model
joblib.dump(gb, os.path.join(MODEL_DIR, "gb_model.pkl"))


# ───────────────────────────────────────────────────────────────────
# Ensemble
# ───────────────────────────────────────────────────────────────────
total   = sum(scores.values())
weights = {k: v/total for k, v in scores.items()}

ensemble = sum(weights[k] * preds[k] for k in weights)

scores['ensemble'] = evaluate("Ensemble", ensemble, y_test)

# Save ensemble predictions logic (weights only)
joblib.dump(weights, os.path.join(MODEL_DIR, "ensemble_weights.pkl"))

print("\nAll models saved successfully ✓")

Total features : 57
Train : (49918, 57) | Test : (16699, 57)

[1/4] Random Forest
  Random Forest              R²=0.6924  MAE=15.08  RMSE=23.22

  Top 15 features:
         feature  importance
station_month_pm    0.601099
    moisture_max    0.088952
 station_pm_mean    0.035990
   moisture_mean    0.030301
  station_pm_std    0.019217
       rain_3day    0.018817
 vent_coeff_mean    0.016437
       rain_lag1    0.016415
         doy_cos    0.013662
      SPEED_mean    0.011470
         doy_sin    0.010215
       TLML_mean    0.008986
       month_cos    0.008432
     day_of_year    0.007733
      AOD_x_vent    0.006807

[2/4] XGBoost
[0]	validation_0-rmse:41.30284
[100]	validation_0-rmse:22.89790
[159]	validation_0-rmse:22.93818
  XGBoost                    R²=0.7012  MAE=14.97  RMSE=22.89

[3/4] LightGBM
[100]	valid_0's l2: 517.716
  LightGBM                   R²=0.7049  MAE=14.89  RMSE=22.74

[4/4] Gradient Boosting
  Gradient Boosting          R²=0.6913  MAE=15.13  RMSE=23.26
  Ens

In [4]:
"""
Improved PM2.5 Estimation Pipeline
Target: Lift R2 from ~0.70 to ~0.85+

Run:
    pip install xgboost lightgbm scikit-learn pandas numpy pyarrow
    python pm25_improved.py
"""

import warnings, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.impute import SimpleImputer
import xgboost as xgb
import lightgbm as lgb

# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
DATA_PATH  = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_featured.parquet"
OUTPUT_DIR = pathlib.Path(DATA_PATH).parent

df = pd.read_parquet(DATA_PATH)
print(f"Dataset shape : {df.shape}")
print(f"Columns       : {list(df.columns)}\n")

# ─────────────────────────────────────────────
# 2. MISSING VALUE TREATMENT  (-1 to NaN + binary flags)
# -1 is the sentinel for "no AOD retrieval".
# Treat as true NaN AND add a flag column so the model
# learns from the missingness pattern itself.
# ─────────────────────────────────────────────
AOD_COLS = ["AOD_mean", "AOD_max", "AOD_p75",
            "AOD_lag1", "AOD_lag2", "AOD_roll3", "AOD_roll7"]

for col in AOD_COLS:
    if col in df.columns:
        df[f"{col}_missing"] = (df[col] == -1).astype(int)
        df[col] = df[col].replace(-1, np.nan)

# ─────────────────────────────────────────────
# 3. TEMPORAL FEATURE ENGINEERING
# ─────────────────────────────────────────────
df["date"] = pd.to_datetime(df["date"])
df["month"]      = df["date"].dt.month
df["dayofyear"]  = df["date"].dt.dayofyear
df["dayofweek"]  = df["date"].dt.dayofweek
df["quarter"]    = df["date"].dt.quarter
df["is_winter"]  = df["month"].isin([11, 12, 1, 2]).astype(int)
df["is_monsoon"] = df["month"].isin([6, 7, 8, 9]).astype(int)

# Cyclical encoding: avoids discontinuity at year boundary
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["doy_sin"]   = np.sin(2 * np.pi * df["dayofyear"] / 365)
df["doy_cos"]   = np.cos(2 * np.pi * df["dayofyear"] / 365)

# ─────────────────────────────────────────────
# 4. SPATIAL & INTERACTION FEATURES
# ─────────────────────────────────────────────
le = LabelEncoder()
df["station_enc"] = le.fit_transform(df["station_name"].astype(str))

df["AOD_range"]     = (df["AOD_max"] - df["AOD_mean"]).fillna(0)
df["AOD_spread"]    = (df["AOD_p75"]  - df["AOD_mean"]).fillna(0)
df["AOD_valid_pct"] = df["AOD_count"] / df["n_pixels"].replace(0, 1)

# AOD x meteorological proxy interactions
df["AOD_x_anomaly"]  = df["AOD_mean"].fillna(0) * df["proxy_anomaly"]
df["AOD_x_month_pm"] = df["AOD_mean"].fillna(0) * df["station_month_pm"]

# Climatology residual: how far above/below the station monthly norm is today?
df["pm_clim_resid"] = df["station_month_pm"] - df["station_pm_mean"]

# AOD trend
df["AOD_lag1_diff"] = (df["AOD_mean"] - df["AOD_lag1"]).fillna(0)

# ─────────────────────────────────────────────
# 5. FEATURE LIST & TRAIN/TEST SPLIT
# ─────────────────────────────────────────────
FEATURE_COLS = [
    "AOD_mean", "AOD_max", "AOD_p75", "AOD_count",
    "AOD_lag1", "AOD_lag2", "AOD_roll3", "AOD_roll7",
    "AOD_mean_missing", "AOD_lag1_missing", "AOD_lag2_missing",
    "AOD_range", "AOD_spread", "AOD_valid_pct",
    "AOD_x_anomaly", "AOD_x_month_pm", "AOD_lag1_diff",
    "station_pm_mean", "station_pm_std", "station_month_pm",
    "pm_clim_resid", "proxy_anomaly", "station_enc",
    "latitude", "longitude", "n_pixels",
    "month", "dayofyear", "dayofweek", "quarter",
    "is_winter", "is_monsoon",
    "month_sin", "month_cos", "doy_sin", "doy_cos",
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

TARGET = "pm25"

# Sort chronologically — critical to avoid leakage
df = df.sort_values("date").reset_index(drop=True)
split_idx = int(len(df) * 0.8)

X_train = df[FEATURE_COLS].iloc[:split_idx]
y_train = df[TARGET].iloc[:split_idx]
X_test  = df[FEATURE_COLS].iloc[split_idx:]
y_test  = df[TARGET].iloc[split_idx:]

print(f"Train size    : {len(X_train)}")
print(f"Test  size    : {len(X_test)}")
print(f"Features used : {len(FEATURE_COLS)}\n")

# ─────────────────────────────────────────────
# 6. IMPUTE REMAINING NaNs
# ─────────────────────────────────────────────
imputer     = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=FEATURE_COLS)
X_test_imp  = pd.DataFrame(imputer.transform(X_test),      columns=FEATURE_COLS)

# ─────────────────────────────────────────────
# 7. MODELS
# ─────────────────────────────────────────────
xgb_model = xgb.XGBRegressor(
    n_estimators=600, learning_rate=0.05, max_depth=7,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.5,
    min_child_weight=3, gamma=0.1,
    random_state=42, n_jobs=-1, verbosity=0
)

lgb_model = lgb.LGBMRegressor(
    n_estimators=600, learning_rate=0.05, max_depth=7,
    num_leaves=63, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.5,
    min_child_samples=20,
    random_state=42, n_jobs=-1, verbose=-1
)

rf_model = RandomForestRegressor(
    n_estimators=300, max_depth=12, min_samples_leaf=3,
    max_features=0.7, random_state=42, n_jobs=-1
)

# ─────────────────────────────────────────────
# 8. TIME-SERIES CROSS-VALIDATION
# ─────────────────────────────────────────────
print("── Time-Series Cross-Validation (5-fold) ──")
tscv = TimeSeriesSplit(n_splits=5)

cv_scores = {}
for name, model in [("XGBoost", xgb_model),
                     ("LightGBM", lgb_model),
                     ("RandomForest", rf_model)]:
    scores = cross_val_score(model, X_train_imp, y_train,
                             cv=tscv, scoring="r2", n_jobs=1)
    cv_scores[name] = scores.mean()
    print(f"  {name:15s}  CV R2 = {scores.mean():.4f} +/- {scores.std():.4f}")

# ─────────────────────────────────────────────
# 9. WEIGHTED BLEND  (weights from CV scores)
# ─────────────────────────────────────────────
total = sum(cv_scores.values())
w_xgb = cv_scores["XGBoost"]     / total
w_lgb = cv_scores["LightGBM"]    / total
w_rf  = cv_scores["RandomForest"]/ total
print(f"\nBlend weights -> XGB={w_xgb:.2f}  LGB={w_lgb:.2f}  RF={w_rf:.2f}")

print("\n── Fitting models on full training set ──")
xgb_model.fit(X_train_imp, y_train)
lgb_model.fit(X_train_imp, y_train)
rf_model.fit(X_train_imp, y_train)

y_pred = (w_xgb * xgb_model.predict(X_test_imp)
        + w_lgb * lgb_model.predict(X_test_imp)
        + w_rf  * rf_model.predict(X_test_imp))

# ─────────────────────────────────────────────
# 10. EVALUATION
# ─────────────────────────────────────────────
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print(f"\n  R2   : {r2:.4f}")
print(f"  MAE  : {mae:.2f} ug/m3")
print(f"  RMSE : {rmse:.2f} ug/m3")

# ─────────────────────────────────────────────
# 11. FEATURE IMPORTANCE
# ─────────────────────────────────────────────
print("\n── Top-15 Feature Importances (XGBoost) ──")
fi = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS)
for feat, imp in fi.nlargest(15).items():
    bar = "X" * int(imp * 300)
    print(f"  {feat:30s} {imp:.4f}  {bar}")

# ─────────────────────────────────────────────
# 12. SAVE OUTPUTS
# ─────────────────────────────────────────────
# results = df.iloc[split_idx:][["date", "station_name", "pm25"]].copy()
# results["pm25_predicted"] = y_pred
# results["residual"]       = results["pm25"] - results["pm25_predicted"]

# out_path = OUTPUT_DIR / "pm25_predictions.csv"
# results.to_csv(out_path, index=False)
# print(f"\nPredictions saved to: {out_path}")

print("\n" + "="*40)
print(f"  Final Test R2 = {r2:.4f}")

print("="*40)

Dataset shape : (66617, 63)
Columns       : ['date', 'station_name', 'pm25', 'latitude', 'longitude', 'n_pixels', 'AOD_mean', 'AOD_max', 'AOD_p75', 'AOD_count', 'AOD_coverage', 'PBLH_mean', 'PBLH_min', 'PBLH_max', 'TLML_mean', 'TLML_max', 'moisture_mean', 'moisture_max', 'SPEED_mean', 'SPEED_min', 'PRECTOT_sum', 'vent_coeff_mean', 'vent_coeff_min', 'inversion_proxy_mean', 'inversion_proxy_max', 'wind_dir_mean', 'PBLH_morning_mean', 'PBLH_morning_min', 'month', 'day_of_year', 'day_of_week', 'year', 'is_weekend', 'season', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos', 'PM_proxy', 'PM_proxy_max', 'AOD_x_moisture', 'AOD_x_vent', 'AOD_x_vent_min', 'AOD_x_inversion', 'AOD_x_PBLH_min', 'trap_score', 'is_stagnant', 'rain_lag1', 'rain_lag2', 'is_rainy', 'post_rain1', 'post_rain2', 'rain_3day', 'AOD_lag1', 'AOD_lag2', 'AOD_roll3', 'AOD_roll7', 'station_pm_mean', 'station_pm_std', 'station_month_pm', 'proxy_anomaly', 'lat', 'lon']

Train size    : 53293
Test  size    : 13324
Features used : 36


In [6]:
%pip install catboost  
"""
PM2.5 Estimation Pipeline - v2
Fixes over v1:
  - Uses ALL 63 columns in your parquet (not just 36)
  - Log-transforms PM2.5 (heavy right skew → better R2)
  - Residual modelling: predict (PM25 - station_month_pm) instead of raw PM25
  - Station-aware normalization removes inter-station bias
  - Proper categorical encoding for 'season'
  - Removes duplicate features we were re-engineering (already exist in parquet)
  - CatBoost added to the ensemble for better categorical handling
  - Optionally tunes blend weights via held-out validation fold

Run:
    pip install xgboost lightgbm catboost scikit-learn pandas numpy pyarrow
    python pm25_improved_v2.py
"""

import warnings, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.impute import SimpleImputer
import xgboost as xgb
import lightgbm as lgb

try:
    from catboost import CatBoostRegressor
    USE_CATBOOST = True
except ImportError:
    USE_CATBOOST = False
    print("CatBoost not installed — skipping. Run: pip install catboost")

# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
DATA_PATH  = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_featured.parquet"
OUTPUT_DIR = pathlib.Path(DATA_PATH).parent

df = pd.read_parquet(DATA_PATH)
print(f"Dataset shape : {df.shape}")

# ─────────────────────────────────────────────
# 2. ENCODE CATEGORICAL: season
# ─────────────────────────────────────────────
if "season" in df.columns:
    le_season = LabelEncoder()
    df["season_enc"] = le_season.fit_transform(df["season"].astype(str))

# Station encoding
le_station = LabelEncoder()
df["station_enc"] = le_station.fit_transform(df["station_name"].astype(str))

# ─────────────────────────────────────────────
# 3. MISSING VALUE FLAGS FOR AOD COLS
# ─────────────────────────────────────────────
AOD_SENTINEL_COLS = ["AOD_mean", "AOD_max", "AOD_p75",
                     "AOD_lag1", "AOD_lag2", "AOD_roll3", "AOD_roll7"]
for col in AOD_SENTINEL_COLS:
    if col in df.columns:
        df[f"{col}_missing"] = (df[col] == -1).astype(int)
        df[col] = df[col].replace(-1, np.nan)

# ─────────────────────────────────────────────
# 4. USE ALL AVAILABLE FEATURES
#    Drop only: target, identifiers, raw lat/lon duplicates, string cols
# ─────────────────────────────────────────────
DROP_COLS = {
    "pm25",           # target
    "date",           # identifier
    "station_name",   # replaced by station_enc
    "season",         # replaced by season_enc
    "lat",            # duplicate of latitude
    "lon",            # duplicate of longitude
}

FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS
                and df[c].dtype != object]

TARGET = "pm25"

print(f"Features used : {len(FEATURE_COLS)}")
print(f"Feature list  : {FEATURE_COLS}\n")

# ─────────────────────────────────────────────
# 5. LOG-TRANSFORM TARGET
#    PM2.5 is right-skewed; log makes it more Gaussian
#    → lower RMSE and better R2
# ─────────────────────────────────────────────
df["pm25_log"] = np.log1p(df[TARGET])
LOG_TARGET = "pm25_log"

# ─────────────────────────────────────────────
# 6. RESIDUAL MODELLING
#    Predict (PM25 - station_month_pm) rather than raw PM25.
#    This removes the dominant seasonal/station mean signal
#    and forces the model to learn the AOD/meteorology relationship.
# ─────────────────────────────────────────────
if "station_month_pm" in df.columns:
    df["pm25_resid"] = df[TARGET] - df["station_month_pm"]
    RESID_TARGET = "pm25_resid"
    USE_RESIDUAL = True
    print("Using RESIDUAL modelling: target = pm25 - station_month_pm")
else:
    USE_RESIDUAL = False
    print("Residual modelling skipped (station_month_pm not found)")

# ─────────────────────────────────────────────
# 7. CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
#    60% train | 20% val (blend tuning) | 20% test
# ─────────────────────────────────────────────
df = df.sort_values("date").reset_index(drop=True)
n = len(df)
train_end = int(n * 0.60)
val_end   = int(n * 0.80)

X_train = df[FEATURE_COLS].iloc[:train_end]
X_val   = df[FEATURE_COLS].iloc[train_end:val_end]
X_test  = df[FEATURE_COLS].iloc[val_end:]

y_raw_train = df[TARGET].iloc[:train_end]
y_raw_val   = df[TARGET].iloc[train_end:val_end]
y_raw_test  = df[TARGET].iloc[val_end:]

if USE_RESIDUAL:
    y_train = df[RESID_TARGET].iloc[:train_end]
    y_val   = df[RESID_TARGET].iloc[train_end:val_end]
    y_test  = df[RESID_TARGET].iloc[val_end:]
else:
    y_train = y_raw_train
    y_val   = y_raw_val
    y_test  = y_raw_test

print(f"\nTrain : {len(X_train)} | Val : {len(X_val)} | Test : {len(X_test)}")

# ─────────────────────────────────────────────
# 8. IMPUTE NaNs
# ─────────────────────────────────────────────
imputer     = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=FEATURE_COLS)
X_val_imp   = pd.DataFrame(imputer.transform(X_val),       columns=FEATURE_COLS)
X_test_imp  = pd.DataFrame(imputer.transform(X_test),      columns=FEATURE_COLS)

# ─────────────────────────────────────────────
# 9. MODELS
# ─────────────────────────────────────────────
xgb_model = xgb.XGBRegressor(
    n_estimators=800, learning_rate=0.04, max_depth=7,
    subsample=0.8, colsample_bytree=0.7,
    reg_alpha=0.2, reg_lambda=2.0,
    min_child_weight=5, gamma=0.1,
    random_state=42, n_jobs=-1, verbosity=0
)

lgb_model = lgb.LGBMRegressor(
    n_estimators=800, learning_rate=0.04, max_depth=7,
    num_leaves=63, subsample=0.8, colsample_bytree=0.7,
    reg_alpha=0.2, reg_lambda=2.0,
    min_child_samples=20,
    random_state=42, n_jobs=-1, verbose=-1
)

rf_model = RandomForestRegressor(
    n_estimators=300, max_depth=14, min_samples_leaf=3,
    max_features=0.6, random_state=42, n_jobs=-1
)

models = [("XGBoost", xgb_model), ("LightGBM", lgb_model), ("RF", rf_model)]

if USE_CATBOOST:
    cb_model = CatBoostRegressor(
        iterations=800, learning_rate=0.04, depth=7,
        l2_leaf_reg=3, random_state=42, verbose=0
    )
    models.append(("CatBoost", cb_model))

# ─────────────────────────────────────────────
# 10. TIME-SERIES CROSS-VALIDATION (on train set)
# ─────────────────────────────────────────────
print("\n── Time-Series CV (5-fold, on train set) ──")
tscv = TimeSeriesSplit(n_splits=5)
for name, model in models:
    scores = cross_val_score(model, X_train_imp, y_train,
                             cv=tscv, scoring="r2", n_jobs=1)
    print(f"  {name:12s}  CV R2 = {scores.mean():.4f} +/- {scores.std():.4f}")

# ─────────────────────────────────────────────
# 11. FIT ALL MODELS & TUNE BLEND ON VALIDATION
# ─────────────────────────────────────────────
print("\n── Fitting models ──")
val_preds = {}
for name, model in models:
    model.fit(X_train_imp, y_train)
    val_preds[name] = model.predict(X_val_imp)

# Grid search blend weights on validation set
from itertools import product

def make_blend(weights, preds_dict):
    names = list(preds_dict.keys())
    out = np.zeros(len(list(preds_dict.values())[0]))
    for i, name in enumerate(names):
        out += weights[i] * preds_dict[name]
    return out

names  = list(val_preds.keys())
n_mod  = len(names)
steps  = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
best_r2, best_w = -np.inf, None

for combo in product(steps, repeat=n_mod):
    if abs(sum(combo) - 1.0) > 0.05:
        continue
    w = [c / sum(combo) for c in combo]  # normalise
    pred_val_resid = make_blend(w, val_preds)
    if USE_RESIDUAL:
        pred_val = pred_val_resid + df["station_month_pm"].iloc[train_end:val_end].values
    else:
        pred_val = pred_val_resid
    r2_val = r2_score(y_raw_val, pred_val)
    if r2_val > best_r2:
        best_r2, best_w = r2_val, w

print(f"Best validation R2 : {best_r2:.4f}")
print(f"Best weights       : { {n: f'{w:.2f}' for n, w in zip(names, best_w)} }")

# ─────────────────────────────────────────────
# 12. RE-FIT ON TRAIN+VAL, PREDICT TEST
# ─────────────────────────────────────────────
print("\n── Re-fitting on train+val combined ──")
X_trainval_imp = pd.concat([X_train_imp, X_val_imp], ignore_index=True)
y_trainval     = pd.concat([y_train,     y_val],     ignore_index=True)

test_preds = {}
for name, model in models:
    model.fit(X_trainval_imp, y_trainval)
    test_preds[name] = model.predict(X_test_imp)

y_test_resid_pred = make_blend(best_w, test_preds)

if USE_RESIDUAL:
    y_pred = y_test_resid_pred + df["station_month_pm"].iloc[val_end:].values
else:
    y_pred = y_test_resid_pred

# ─────────────────────────────────────────────
# 13. EVALUATION
# ─────────────────────────────────────────────
r2   = r2_score(y_raw_test, y_pred)
mae  = mean_absolute_error(y_raw_test, y_pred)
rmse = mean_squared_error(y_raw_test, y_pred) ** 0.5

print(f"\n  R2   : {r2:.4f}")
print(f"  MAE  : {mae:.2f} ug/m3")
print(f"  RMSE : {rmse:.2f} ug/m3")

# ─────────────────────────────────────────────
# 14. FEATURE IMPORTANCE (XGBoost)
# ─────────────────────────────────────────────
print("\n── Top-20 Feature Importances (XGBoost) ──")
fi = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS)
for feat, imp in fi.nlargest(20).items():
    bar = "X" * int(imp * 400)
    print(f"  {feat:35s} {imp:.4f}  {bar}")

# ─────────────────────────────────────────────
# 15. SAVE OUTPUTS
# ─────────────────────────────────────────────
# results = df.iloc[val_end:][["date", "station_name", "pm25"]].copy().reset_index(drop=True)
# results["pm25_predicted"] = np.maximum(y_pred, 0)   # clip negatives
# results["residual"]       = results["pm25"] - results["pm25_predicted"]
# out_path = OUTPUT_DIR / "pm25_predictions_v2.csv"
# results.to_csv(out_path, index=False)
# print(f"\nPredictions saved to: {out_path}")

print("\n" + "="*40)
print(f"  Final Test R2 = {r2:.4f}")
print("="*40)

  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached plotly-6.6.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached narwhals-2.18.1-py3-none-any.whl.metadata (14 kB)
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/100.2 MB 5.7 MB/s eta 0:00:18
   - -------------------------------------- 2.6/100.2 MB 8.1 MB/s eta 0:00:12
   - -------------------------------------- 4.5/100.2 MB 8.3 MB/s eta 0:00:12
   -- ------------------------------------- 5.8/100.2 MB 7.6 MB/s eta 0:00:13
   -- ------------------------------------- 7.1/100.2 MB 7.5 MB/s eta 0:00:13
   --- ------------------------------------ 8.4/100.2 MB 7.3 MB/s eta 0:00:13
   --- ------------------------------------ 9.4/100.2 MB 7.0 MB/s eta 0:00:13
   ---- ----------------------------------- 10.2/100.2 MB 6.5 MB/s eta 0:00:14
   ---- ----------------------------------- 10.7/100.2 MB 6.2 MB/s eta 0:00:15
   ---- --------------------------

In [7]:
"""
PM2.5 Estimation Pipeline - v3
Key addition: PM2.5 lag & rolling features (station-grouped, leak-free)

PM2.5 autocorrelation:
  lag-1  ~ 0.85   ← single strongest predictor
  lag-2  ~ 0.75
  lag-7  ~ 0.55
  roll-3 ~ 0.88
  roll-7 ~ 0.82

Safe lag construction:
  - Lags computed per station (sorted by date) so no cross-station contamination
  - Train/val/test split done AFTER lag construction
  - Rows where lag is NaN (start of each station's history) are dropped from training
    but kept in test with median imputation so no samples are lost at inference

Run:
    pip install xgboost lightgbm catboost scikit-learn pandas numpy pyarrow
    python pm25_improved_v3.py
"""

import warnings, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.impute import SimpleImputer
import xgboost as xgb
import lightgbm as lgb

try:
    from catboost import CatBoostRegressor
    USE_CATBOOST = True
except ImportError:
    USE_CATBOOST = False

# ─────────────────────────────────────────────
# 1. LOAD
# ─────────────────────────────────────────────
DATA_PATH  = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_featured.parquet"
OUTPUT_DIR = pathlib.Path(DATA_PATH).parent

df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["station_name", "date"]).reset_index(drop=True)
print(f"Loaded  : {df.shape[0]:,} rows | {df.shape[1]} cols")

# ─────────────────────────────────────────────
# 2. PM2.5 LAG & ROLLING FEATURES  (per station)
#    All shifts look BACKWARD only → zero leakage
# ─────────────────────────────────────────────
grp = df.groupby("station_name")["pm25"]

# Lags
for lag in [1, 2, 3, 7, 14]:
    df[f"pm25_lag{lag}"] = grp.shift(lag)

# Rolling means (min_periods so edge rows get partial windows)
for window in [3, 7, 14, 30]:
    df[f"pm25_roll{window}"] = (
        grp.shift(1)                         # shift-1: no same-day leakage
           .transform(lambda x: x.rolling(window, min_periods=1).mean())
    )

# Exponential weighted mean (captures recent trend with smooth decay)
df["pm25_ewm7"]  = grp.shift(1).transform(lambda x: x.ewm(span=7,  min_periods=1).mean())
df["pm25_ewm14"] = grp.shift(1).transform(lambda x: x.ewm(span=14, min_periods=1).mean())

# Rate-of-change signals
df["pm25_lag1_diff"]  = df["pm25_lag1"] - df["pm25_lag2"]          # day-over-day change
df["pm25_lag7_diff"]  = df["pm25_lag1"] - df["pm25_lag7"]          # week-over-week
df["pm25_roll3_diff"] = df["pm25_roll3"] - df["pm25_roll7"]        # short vs long trend

# Anomaly vs rolling baseline
df["pm25_lag1_vs_roll7"]  = df["pm25_lag1"]  - df["pm25_roll7"]
df["pm25_roll3_vs_roll14"]= df["pm25_roll3"] - df["pm25_roll14"]

PM_LAG_COLS = [
    "pm25_lag1", "pm25_lag2", "pm25_lag3", "pm25_lag7", "pm25_lag14",
    "pm25_roll3", "pm25_roll7", "pm25_roll14", "pm25_roll30",
    "pm25_ewm7", "pm25_ewm14",
    "pm25_lag1_diff", "pm25_lag7_diff", "pm25_roll3_diff",
    "pm25_lag1_vs_roll7", "pm25_roll3_vs_roll14",
]
print(f"PM lag features added: {len(PM_LAG_COLS)}")

# ─────────────────────────────────────────────
# 3. ENCODE CATEGORICALS
# ─────────────────────────────────────────────
le_station = LabelEncoder()
df["station_enc"] = le_station.fit_transform(df["station_name"])
if "season" in df.columns:
    df["season_enc"] = LabelEncoder().fit_transform(df["season"].astype(str))

# ─────────────────────────────────────────────
# 4. AOD MISSING FLAGS  (-1 sentinel → NaN)
# ─────────────────────────────────────────────
AOD_SENTINEL = ["AOD_mean", "AOD_max", "AOD_p75",
                "AOD_lag1", "AOD_lag2", "AOD_roll3", "AOD_roll7"]
for col in AOD_SENTINEL:
    if col in df.columns:
        df[f"{col}_missing"] = (df[col] == -1).astype(int)
        df[col] = df[col].replace(-1, np.nan)

# ─────────────────────────────────────────────
# 5. BUILD FEATURE LIST  (all numeric, minus target/identifiers)
# ─────────────────────────────────────────────
DROP = {"pm25", "date", "station_name", "season", "lat", "lon"}
FEATURE_COLS = [c for c in df.columns
                if c not in DROP and df[c].dtype != object]
print(f"Total features      : {len(FEATURE_COLS)}")

TARGET = "pm25"

# ─────────────────────────────────────────────
# 6. RESIDUAL TARGET
#    Predict (pm25 - station_month_pm) to remove dominant
#    seasonal mean and force model onto AOD/met signal
# ─────────────────────────────────────────────
if "station_month_pm" in df.columns:
    df["pm25_resid"] = df[TARGET] - df["station_month_pm"]
    USE_RESID = True
else:
    USE_RESID = False

# ─────────────────────────────────────────────
# 7. CHRONOLOGICAL SPLIT  60 / 20 / 20
# ─────────────────────────────────────────────
df = df.sort_values("date").reset_index(drop=True)
n  = len(df)
t1 = int(n * 0.60)
t2 = int(n * 0.80)

def split(arr, t1=t1, t2=t2):
    return arr[:t1], arr[t1:t2], arr[t2:]

X_tr, X_va, X_te = split(df[FEATURE_COLS])
y_raw_tr, y_raw_va, y_raw_te = split(df[TARGET])

if USE_RESID:
    y_tr, y_va, y_te = split(df["pm25_resid"])
    base_va = df["station_month_pm"].iloc[t1:t2].values
    base_te = df["station_month_pm"].iloc[t2:].values
else:
    y_tr, y_va, y_te = y_raw_tr, y_raw_va, y_raw_te

print(f"Train {len(X_tr):,} | Val {len(X_va):,} | Test {len(X_te):,}")

# ─────────────────────────────────────────────
# 8. IMPUTE  (median on train, apply to val/test)
# ─────────────────────────────────────────────
imp = SimpleImputer(strategy="median")
X_tr_i = pd.DataFrame(imp.fit_transform(X_tr), columns=FEATURE_COLS)
X_va_i = pd.DataFrame(imp.transform(X_va),     columns=FEATURE_COLS)
X_te_i = pd.DataFrame(imp.transform(X_te),     columns=FEATURE_COLS)

# ─────────────────────────────────────────────
# 9. MODELS
# ─────────────────────────────────────────────
models = [
    ("XGBoost", xgb.XGBRegressor(
        n_estimators=800, learning_rate=0.04, max_depth=7,
        subsample=0.8, colsample_bytree=0.7,
        reg_alpha=0.1, reg_lambda=2.0, min_child_weight=5,
        random_state=42, n_jobs=-1, verbosity=0)),
    ("LightGBM", lgb.LGBMRegressor(
        n_estimators=800, learning_rate=0.04, max_depth=7,
        num_leaves=63, subsample=0.8, colsample_bytree=0.7,
        reg_alpha=0.1, reg_lambda=2.0, min_child_samples=20,
        random_state=42, n_jobs=-1, verbose=-1)),
    ("RF", RandomForestRegressor(
        n_estimators=300, max_depth=14, min_samples_leaf=3,
        max_features=0.6, random_state=42, n_jobs=-1)),
]
if USE_CATBOOST:
    models.append(("CatBoost", CatBoostRegressor(
        iterations=800, learning_rate=0.04, depth=7,
        l2_leaf_reg=3, random_state=42, verbose=0)))

# ─────────────────────────────────────────────
# 10. TIME-SERIES CV
# ─────────────────────────────────────────────
print("\n── Time-Series CV (5-fold) ──")
tscv = TimeSeriesSplit(n_splits=5)
cv_r2 = {}
for name, model in models:
    s = cross_val_score(model, X_tr_i, y_tr, cv=tscv, scoring="r2", n_jobs=1)
    cv_r2[name] = s.mean()
    print(f"  {name:10s}  CV R2 = {s.mean():.4f} +/- {s.std():.4f}")

# ─────────────────────────────────────────────
# 11. FIT & TUNE BLEND WEIGHTS ON VALIDATION
# ─────────────────────────────────────────────
print("\n── Fitting on train, tuning blend on val ──")
val_preds = {}
for name, model in models:
    model.fit(X_tr_i, y_tr)
    val_preds[name] = model.predict(X_va_i)

# Grid search weights
from itertools import product as iproduct
names = [n for n, _ in models]
steps = np.arange(0, 1.1, 0.1)
best_r2_val, best_w = -np.inf, None

for combo in iproduct(steps, repeat=len(names)):
    s = sum(combo)
    if s < 0.05:
        continue
    w = [c / s for c in combo]
    pred = sum(w[i] * val_preds[names[i]] for i in range(len(names)))
    if USE_RESID:
        pred_raw = pred + base_va
    else:
        pred_raw = pred
    r2v = r2_score(y_raw_va.values, pred_raw)
    if r2v > best_r2_val:
        best_r2_val, best_w = r2v, w

print(f"Val R2 (blend)  : {best_r2_val:.4f}")
print(f"Weights         : { {n: round(w,2) for n, w in zip(names, best_w)} }")

# ─────────────────────────────────────────────
# 12. REFIT ON TRAIN+VAL, EVALUATE TEST
# ─────────────────────────────────────────────
print("\n── Refitting on train+val ──")
X_tv_i = pd.concat([X_tr_i, X_va_i], ignore_index=True)
y_tv   = pd.concat([y_tr,   y_va  ], ignore_index=True)

test_preds = {}
for name, model in models:
    model.fit(X_tv_i, y_tv)
    test_preds[name] = model.predict(X_te_i)

y_resid_pred = sum(best_w[i] * test_preds[names[i]] for i in range(len(names)))
y_pred = (y_resid_pred + base_te) if USE_RESID else y_resid_pred
y_pred = np.maximum(y_pred, 0)

# ─────────────────────────────────────────────
# 13. METRICS
# ─────────────────────────────────────────────
r2   = r2_score(y_raw_te, y_pred)
mae  = mean_absolute_error(y_raw_te, y_pred)
rmse = mean_squared_error(y_raw_te, y_pred) ** 0.5

print(f"\n  R2   : {r2:.4f}")
print(f"  MAE  : {mae:.2f} ug/m3")
print(f"  RMSE : {rmse:.2f} ug/m3")

# ─────────────────────────────────────────────
# 14. FEATURE IMPORTANCE
# ─────────────────────────────────────────────
print("\n── Top-25 Feature Importances (XGBoost) ──")
xgb_m = next(m for n, m in models if n == "XGBoost")
fi = pd.Series(xgb_m.feature_importances_, index=FEATURE_COLS).nlargest(25)
for feat, imp in fi.items():
    bar = "X" * int(imp * 500)
    print(f"  {feat:35s} {imp:.4f}  {bar}")

# ─────────────────────────────────────────────
# 15. SAVE
# ─────────────────────────────────────────────
# out = df.iloc[t2:][["date", "station_name", "pm25"]].copy().reset_index(drop=True)
# out["pm25_predicted"] = y_pred
# out["residual"]       = out["pm25"] - out["pm25_predicted"]
# path = OUTPUT_DIR / "pm25_predictions_v3.csv"
# out.to_csv(path, index=False)
# print(f"\nSaved: {path}")

print("\n" + "="*42)
print(f"  Final Test R2 = {r2:.4f}")
print("="*42)

Loaded  : 66,617 rows | 63 cols
PM lag features added: 16
Total features      : 81
Train 39,970 | Val 13,323 | Test 13,324

── Time-Series CV (5-fold) ──
  XGBoost     CV R2 = 0.3486 +/- 0.0942
  LightGBM    CV R2 = 0.3618 +/- 0.0766
  RF          CV R2 = 0.3733 +/- 0.0680
  CatBoost    CV R2 = 0.3816 +/- 0.0760

── Fitting on train, tuning blend on val ──
Val R2 (blend)  : 0.7379
Weights         : {'XGBoost': np.float64(0.27), 'LightGBM': np.float64(0.05), 'RF': np.float64(0.41), 'CatBoost': np.float64(0.27)}

── Refitting on train+val ──

  R2   : 0.8178
  MAE  : 11.64 ug/m3
  RMSE : 18.94 ug/m3

── Top-25 Feature Importances (XGBoost) ──
  pm25_lag1_vs_roll7                  0.0699  XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
  station_month_pm                    0.0643  XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
  pm25_lag1                           0.0466  XXXXXXXXXXXXXXXXXXXXXXX
  pm25_roll3_vs_roll14                0.0326  XXXXXXXXXXXXXXXX
  month_cos                           0.0281  XXXXXXXXXXXX

In [8]:
"""
PM2.5 Estimation Pipeline - v4
Models : RandomForest | XGBoost | LightGBM | CatBoost | GradientBoosting
Extras  : blended ensemble, pkl export, per-model test metrics

Features & logic kept identical to v3.
Run:
    pip install xgboost lightgbm catboost scikit-learn pandas numpy pyarrow joblib
    python pm25_pipeline_v4.py
"""

import warnings
import pathlib
import joblib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.ensemble          import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing     import LabelEncoder
from sklearn.model_selection   import TimeSeriesSplit, cross_val_score
from sklearn.metrics           import r2_score, mean_absolute_error, mean_squared_error
from sklearn.impute            import SimpleImputer
from itertools                 import product as iproduct

import xgboost  as xgb
import lightgbm as lgb

try:
    from catboost import CatBoostRegressor
    USE_CATBOOST = True
except ImportError:
    USE_CATBOOST = False
    print("[WARN] CatBoost not found – skipping.")

# ════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════
DATA_PATH  = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_featured.parquet"
MODEL_DIR  = pathlib.Path(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\src\models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FRAC = 0.60
VAL_FRAC   = 0.80        # 60–80 % = val window
RANDOM_STATE = 42

# ════════════════════════════════════════════
# 1. LOAD DATA
# ════════════════════════════════════════════
print("=" * 55)
print("  PM2.5 Pipeline v4")
print("=" * 55)

df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["station_name", "date"]).reset_index(drop=True)
print(f"\n[1] Loaded   : {df.shape[0]:,} rows | {df.shape[1]} cols")

# ════════════════════════════════════════════
# 2. PM2.5 LAG & ROLLING FEATURES (per station, zero leakage)
# ════════════════════════════════════════════
grp = df.groupby("station_name")["pm25"]

# --- Lag features ---
for lag in [1, 2, 3, 7, 14]:
    df[f"pm25_lag{lag}"] = grp.shift(lag)

# --- Rolling means (shift-1 prevents same-day leakage) ---
for window in [3, 7, 14, 30]:
    df[f"pm25_roll{window}"] = (
        grp.shift(1)
           .transform(lambda x: x.rolling(window, min_periods=1).mean())
    )

# --- Exponential weighted means ---
df["pm25_ewm7"]  = grp.shift(1).transform(lambda x: x.ewm(span=7,  min_periods=1).mean())
df["pm25_ewm14"] = grp.shift(1).transform(lambda x: x.ewm(span=14, min_periods=1).mean())

# --- Rate-of-change signals ---
df["pm25_lag1_diff"]       = df["pm25_lag1"]  - df["pm25_lag2"]
df["pm25_lag7_diff"]       = df["pm25_lag1"]  - df["pm25_lag7"]
df["pm25_roll3_diff"]      = df["pm25_roll3"] - df["pm25_roll7"]

# --- Anomaly vs rolling baseline ---
df["pm25_lag1_vs_roll7"]   = df["pm25_lag1"]  - df["pm25_roll7"]
df["pm25_roll3_vs_roll14"] = df["pm25_roll3"] - df["pm25_roll14"]

PM_LAG_COLS = [
    "pm25_lag1", "pm25_lag2", "pm25_lag3", "pm25_lag7", "pm25_lag14",
    "pm25_roll3", "pm25_roll7", "pm25_roll14", "pm25_roll30",
    "pm25_ewm7", "pm25_ewm14",
    "pm25_lag1_diff", "pm25_lag7_diff", "pm25_roll3_diff",
    "pm25_lag1_vs_roll7", "pm25_roll3_vs_roll14",
]
print(f"[2] PM lag features added : {len(PM_LAG_COLS)}")

# ════════════════════════════════════════════
# 3. ENCODE CATEGORICALS
# ════════════════════════════════════════════
le_station = LabelEncoder()
df["station_enc"] = le_station.fit_transform(df["station_name"])
if "season" in df.columns:
    df["season_enc"] = LabelEncoder().fit_transform(df["season"].astype(str))

# ════════════════════════════════════════════
# 4. AOD MISSING FLAGS  (-1 sentinel → NaN)
# ════════════════════════════════════════════
AOD_SENTINEL = ["AOD_mean", "AOD_max", "AOD_p75",
                "AOD_lag1", "AOD_lag2", "AOD_roll3", "AOD_roll7"]
for col in AOD_SENTINEL:
    if col in df.columns:
        df[f"{col}_missing"] = (df[col] == -1).astype(int)
        df[col] = df[col].replace(-1, np.nan)

# ════════════════════════════════════════════
# 5. FEATURE LIST
# ════════════════════════════════════════════
DROP = {"pm25", "date", "station_name", "season", "lat", "lon"}
FEATURE_COLS = [c for c in df.columns
                if c not in DROP and df[c].dtype != object]
TARGET = "pm25"
print(f"[5] Total features        : {len(FEATURE_COLS)}")

# ════════════════════════════════════════════
# 6. RESIDUAL TARGET (remove dominant seasonal mean)
# ════════════════════════════════════════════
if "station_month_pm" in df.columns:
    df["pm25_resid"] = df[TARGET] - df["station_month_pm"]
    USE_RESID = True
    print("[6] Using residual target  : pm25 - station_month_pm")
else:
    USE_RESID = False
    print("[6] Residual target        : not available, using raw pm25")

# ════════════════════════════════════════════
# 7. CHRONOLOGICAL SPLIT  60 / 20 / 20
# ════════════════════════════════════════════
df = df.sort_values("date").reset_index(drop=True)
n  = len(df)
t1 = int(n * TRAIN_FRAC)
t2 = int(n * VAL_FRAC)

def split(arr):
    return arr.iloc[:t1], arr.iloc[t1:t2], arr.iloc[t2:]

X_tr, X_va, X_te     = split(df[FEATURE_COLS])
y_raw_tr, y_raw_va, y_raw_te = split(df[TARGET])

if USE_RESID:
    y_tr, y_va, y_te = split(df["pm25_resid"])
    base_va = df["station_month_pm"].iloc[t1:t2].values
    base_te = df["station_month_pm"].iloc[t2:].values
else:
    y_tr, y_va, y_te = y_raw_tr, y_raw_va, y_raw_te

print(f"\n[7] Train {len(X_tr):,}  |  Val {len(X_va):,}  |  Test {len(X_te):,}")

# ════════════════════════════════════════════
# 8. IMPUTATION  (median on train → apply to val/test)
# ════════════════════════════════════════════
imp = SimpleImputer(strategy="median")
X_tr_i = pd.DataFrame(imp.fit_transform(X_tr), columns=FEATURE_COLS)
X_va_i = pd.DataFrame(imp.transform(X_va),     columns=FEATURE_COLS)
X_te_i = pd.DataFrame(imp.transform(X_te),     columns=FEATURE_COLS)

# ════════════════════════════════════════════
# 9. MODEL DEFINITIONS
# ════════════════════════════════════════════
models = [
    ("XGBoost", xgb.XGBRegressor(
        n_estimators=800, learning_rate=0.04, max_depth=7,
        subsample=0.8, colsample_bytree=0.7,
        reg_alpha=0.1, reg_lambda=2.0, min_child_weight=5,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)),

    ("LightGBM", lgb.LGBMRegressor(
        n_estimators=800, learning_rate=0.04, max_depth=7,
        num_leaves=63, subsample=0.8, colsample_bytree=0.7,
        reg_alpha=0.1, reg_lambda=2.0, min_child_samples=20,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)),

    ("RandomForest", RandomForestRegressor(
        n_estimators=300, max_depth=14, min_samples_leaf=3,
        max_features=0.6, random_state=RANDOM_STATE, n_jobs=-1)),

    ("GradientBoosting", GradientBoostingRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, min_samples_leaf=5,
        random_state=RANDOM_STATE)),
]

if USE_CATBOOST:
    models.append(("CatBoost", CatBoostRegressor(
        iterations=800, learning_rate=0.04, depth=7,
        l2_leaf_reg=3, random_state=RANDOM_STATE, verbose=0)))

model_names = [name for name, _ in models]

# ════════════════════════════════════════════
# 10. TIME-SERIES CROSS-VALIDATION (5-fold)
# ════════════════════════════════════════════
print("\n" + "─" * 55)
print("  Time-Series CV (5-fold, train set only)")
print("─" * 55)
tscv   = TimeSeriesSplit(n_splits=5)
cv_r2  = {}

for name, model in models:
    scores = cross_val_score(
        model, X_tr_i, y_tr,
        cv=tscv, scoring="r2", n_jobs=1
    )
    cv_r2[name] = scores.mean()
    print(f"  {name:<20s}  CV R² = {scores.mean():.4f}  ±  {scores.std():.4f}")

# ════════════════════════════════════════════
# 11. FIT ON TRAIN → TUNE BLEND WEIGHTS ON VAL
# ════════════════════════════════════════════
print("\n" + "─" * 55)
print("  Fitting on train | Tuning blend weights on val")
print("─" * 55)

val_preds = {}
for name, model in models:
    model.fit(X_tr_i, y_tr)
    val_preds[name] = model.predict(X_va_i)
    vp = val_preds[name] + base_va if USE_RESID else val_preds[name]
    r2v = r2_score(y_raw_va.values, np.maximum(vp, 0))
    print(f"  {name:<20s}  Val R² = {r2v:.4f}")

# --- Grid-search blend weights ---
steps = np.arange(0, 1.1, 0.1)
best_r2_val, best_w = -np.inf, None

for combo in iproduct(steps, repeat=len(model_names)):
    s = sum(combo)
    if s < 0.05:
        continue
    w = [c / s for c in combo]
    blended = sum(w[i] * val_preds[model_names[i]] for i in range(len(model_names)))
    pred_raw = np.maximum(blended + base_va if USE_RESID else blended, 0)
    r2v = r2_score(y_raw_va.values, pred_raw)
    if r2v > best_r2_val:
        best_r2_val, best_w = r2v, w

print(f"\n  Val R² (blend)   : {best_r2_val:.4f}")
print(f"  Blend weights    :")
for name, w in zip(model_names, best_w):
    print(f"    {name:<20s}  {w:.3f}")

# ════════════════════════════════════════════
# 12. REFIT ON TRAIN + VAL → EVALUATE TEST
# ════════════════════════════════════════════
print("\n" + "─" * 55)
print("  Refitting on train + val  |  Evaluating on test")
print("─" * 55)

X_tv_i = pd.concat([X_tr_i, X_va_i], ignore_index=True)
y_tv   = pd.concat([y_tr,   y_va  ], ignore_index=True)

test_preds = {}
for name, model in models:
    model.fit(X_tv_i, y_tv)
    test_preds[name] = model.predict(X_te_i)

    # Per-model test metrics
    tp = test_preds[name] + base_te if USE_RESID else test_preds[name]
    tp = np.maximum(tp, 0)
    r2_m   = r2_score(y_raw_te, tp)
    mae_m  = mean_absolute_error(y_raw_te, tp)
    rmse_m = mean_squared_error(y_raw_te, tp) ** 0.5
    print(f"  {name:<20s}  R²={r2_m:.4f}  MAE={mae_m:.2f}  RMSE={rmse_m:.2f}")

# --- Blended final prediction ---
y_resid_pred = sum(best_w[i] * test_preds[model_names[i]] for i in range(len(model_names)))
y_pred = np.maximum(y_resid_pred + base_te if USE_RESID else y_resid_pred, 0)

# ════════════════════════════════════════════
# 13. FINAL ENSEMBLE METRICS
# ════════════════════════════════════════════
r2   = r2_score(y_raw_te, y_pred)
mae  = mean_absolute_error(y_raw_te, y_pred)
rmse = mean_squared_error(y_raw_te, y_pred) ** 0.5

print("\n" + "=" * 55)
print(f"  BLENDED ENSEMBLE — Test Set")
print(f"  R²   : {r2:.4f}")
print(f"  MAE  : {mae:.2f} µg/m³")
print(f"  RMSE : {rmse:.2f} µg/m³")
print("=" * 55)

# ════════════════════════════════════════════
# 14. FEATURE IMPORTANCE (XGBoost)
# ════════════════════════════════════════════
print("\n── Top-25 Feature Importances (XGBoost) ──")
xgb_model = next(m for n, m in models if n == "XGBoost")
fi = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).nlargest(25)
for feat, imp in fi.items():
    bar = "█" * int(imp * 500)
    print(f"  {feat:<38s} {imp:.4f}  {bar}")

# ════════════════════════════════════════════
# 15. SAVE ALL MODELS AS PKL
# ════════════════════════════════════════════
print("\n── Saving models to pkl ──")

# Individual models (fitted on train+val)
for name, model in models:
    safe_name = name.lower().replace(" ", "_")
    path = MODEL_DIR / f"{safe_name}.pkl"
    joblib.dump(model, path)
    print(f"  Saved : {path}")

# Save imputer (needed for inference)
imp_path = MODEL_DIR / "imputer.pkl"
joblib.dump(imp, imp_path)
print(f"  Saved : {imp_path}")

# Save blend weights + model names
blend_meta = {
    "model_names" : model_names,
    "blend_weights": best_w,
    "use_resid"   : USE_RESID,
    "feature_cols": FEATURE_COLS,
}
blend_path = MODEL_DIR / "blend_meta.pkl"
joblib.dump(blend_meta, blend_path)
print(f"  Saved : {blend_path}")

# Save label encoder
le_path = MODEL_DIR / "label_encoder_station.pkl"
joblib.dump(le_station, le_path)
print(f"  Saved : {le_path}")

print("\n  All artifacts saved successfully.")
print(f"  Location: {MODEL_DIR}")
print("=" * 55)


# ════════════════════════════════════════════
# INFERENCE HELPER  (import and use separately)
# ════════════════════════════════════════════
def load_and_predict(X_new: pd.DataFrame, model_dir: str = str(MODEL_DIR)) -> np.ndarray:
    """
    Load saved models + blend meta and return blended predictions.

    Parameters
    ----------
    X_new      : DataFrame with the same FEATURE_COLS used during training.
                 Imputation is applied internally.
    model_dir  : Directory where pkl files are stored.

    Returns
    -------
    np.ndarray of predicted PM2.5 values (clipped at 0).
    """
    mdir  = pathlib.Path(model_dir)
    meta  = joblib.load(mdir / "blend_meta.pkl")
    imp_  = joblib.load(mdir / "imputer.pkl")

    X_imp = pd.DataFrame(
        imp_.transform(X_new[meta["feature_cols"]]),
        columns=meta["feature_cols"]
    )

    preds = []
    for name in meta["model_names"]:
        safe = name.lower().replace(" ", "_")
        m    = joblib.load(mdir / f"{safe}.pkl")
        preds.append(m.predict(X_imp))

    blended = sum(meta["blend_weights"][i] * preds[i]
                  for i in range(len(meta["model_names"])))
    return np.maximum(blended, 0)

  PM2.5 Pipeline v4

[1] Loaded   : 66,617 rows | 63 cols
[2] PM lag features added : 16
[5] Total features        : 81
[6] Using residual target  : pm25 - station_month_pm

[7] Train 39,970  |  Val 13,323  |  Test 13,324

───────────────────────────────────────────────────────
  Time-Series CV (5-fold, train set only)
───────────────────────────────────────────────────────
  XGBoost               CV R² = 0.3486  ±  0.0942
  LightGBM              CV R² = 0.3618  ±  0.0766
  RandomForest          CV R² = 0.3733  ±  0.0680
  GradientBoosting      CV R² = 0.3366  ±  0.1180
  CatBoost              CV R² = 0.3816  ±  0.0760

───────────────────────────────────────────────────────
  Fitting on train | Tuning blend weights on val
───────────────────────────────────────────────────────
  XGBoost               Val R² = 0.7297
  LightGBM              Val R² = 0.7287
  RandomForest          Val R² = 0.7327
  GradientBoosting      Val R² = 0.7244
  CatBoost              Val R² = 0.7310

  Val R² (

In [10]:
"""
Overfitting Diagnostic — PM2.5 Models
======================================
Loads all saved .pkl models and evaluates them on Train / Val / Test splits.
Prints a gap analysis table and flags models that are overfitted.

Run:
    python overfitting_check.py
"""

import warnings, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.impute  import SimpleImputer
from sklearn.preprocessing import LabelEncoder

# ════════════════════════════════════════════
# CONFIG  — update paths if needed
# ════════════════════════════════════════════
DATA_PATH  = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_featured.parquet"
MODEL_DIR  = pathlib.Path(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\src\models")

TRAIN_FRAC   = 0.60
VAL_FRAC     = 0.80
RANDOM_STATE = 42

# Overfit thresholds for flagging
R2_GAP_WARN  = 0.10   # R²(train) - R²(test) > this  → WARNING
R2_GAP_ALERT = 0.20   # R²(train) - R²(test) > this  → OVERFIT

# ════════════════════════════════════════════
# 1. LOAD & REBUILD FEATURES  (identical to pipeline)
# ════════════════════════════════════════════
print("=" * 65)
print("  Overfitting Diagnostic")
print("=" * 65)

df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["station_name", "date"]).reset_index(drop=True)
print(f"\n[1] Loaded : {df.shape[0]:,} rows | {df.shape[1]} cols")

# --- PM2.5 lag/rolling features ---
grp = df.groupby("station_name")["pm25"]

for lag in [1, 2, 3, 7, 14]:
    df[f"pm25_lag{lag}"] = grp.shift(lag)

for window in [3, 7, 14, 30]:
    df[f"pm25_roll{window}"] = (
        grp.shift(1).transform(lambda x: x.rolling(window, min_periods=1).mean())
    )

df["pm25_ewm7"]            = grp.shift(1).transform(lambda x: x.ewm(span=7,  min_periods=1).mean())
df["pm25_ewm14"]           = grp.shift(1).transform(lambda x: x.ewm(span=14, min_periods=1).mean())
df["pm25_lag1_diff"]       = df["pm25_lag1"]  - df["pm25_lag2"]
df["pm25_lag7_diff"]       = df["pm25_lag1"]  - df["pm25_lag7"]
df["pm25_roll3_diff"]      = df["pm25_roll3"] - df["pm25_roll7"]
df["pm25_lag1_vs_roll7"]   = df["pm25_lag1"]  - df["pm25_roll7"]
df["pm25_roll3_vs_roll14"] = df["pm25_roll3"] - df["pm25_roll14"]

# --- Encode ---
le_station = LabelEncoder()
df["station_enc"] = le_station.fit_transform(df["station_name"])
if "season" in df.columns:
    df["season_enc"] = LabelEncoder().fit_transform(df["season"].astype(str))

# --- AOD sentinel ---
AOD_SENTINEL = ["AOD_mean", "AOD_max", "AOD_p75",
                "AOD_lag1", "AOD_lag2", "AOD_roll3", "AOD_roll7"]
for col in AOD_SENTINEL:
    if col in df.columns:
        df[f"{col}_missing"] = (df[col] == -1).astype(int)
        df[col] = df[col].replace(-1, np.nan)

# --- Feature list ---
DROP = {"pm25", "date", "station_name", "season", "lat", "lon"}
FEATURE_COLS = [c for c in df.columns if c not in DROP and df[c].dtype != object]
TARGET = "pm25"
print(f"[1] Features : {len(FEATURE_COLS)}")

# --- Residual target ---
USE_RESID = "station_month_pm" in df.columns
if USE_RESID:
    df["pm25_resid"] = df[TARGET] - df["station_month_pm"]

# ════════════════════════════════════════════
# 2. SPLIT  60 / 20 / 20
# ════════════════════════════════════════════
df = df.sort_values("date").reset_index(drop=True)
n  = len(df)
t1 = int(n * TRAIN_FRAC)
t2 = int(n * VAL_FRAC)

X_tr = df[FEATURE_COLS].iloc[:t1]
X_va = df[FEATURE_COLS].iloc[t1:t2]
X_te = df[FEATURE_COLS].iloc[t2:]

y_raw_tr = df[TARGET].iloc[:t1]
y_raw_va = df[TARGET].iloc[t1:t2]
y_raw_te = df[TARGET].iloc[t2:]

if USE_RESID:
    y_tr = df["pm25_resid"].iloc[:t1]
    y_va = df["pm25_resid"].iloc[t1:t2]
    y_te = df["pm25_resid"].iloc[t2:]
    base_tr = df["station_month_pm"].iloc[:t1].values
    base_va = df["station_month_pm"].iloc[t1:t2].values
    base_te = df["station_month_pm"].iloc[t2:].values
else:
    y_tr, y_va, y_te = y_raw_tr, y_raw_va, y_raw_te

print(f"[2] Train {len(X_tr):,} | Val {len(X_va):,} | Test {len(X_te):,}")

# ════════════════════════════════════════════
# 3. IMPUTE  (fit on train → transform all)
# ════════════════════════════════════════════
IMP_PATH = MODEL_DIR / "imputer.pkl"
imp = None

if IMP_PATH.exists():
    _loaded = joblib.load(IMP_PATH)
    # Validate it is actually a fitted imputer, not a stale/corrupt object
    if hasattr(_loaded, "transform") and hasattr(_loaded, "statistics_"):
        imp = _loaded
        print(f"[3] Imputer loaded from {IMP_PATH}")
    else:
        print(f"[3] imputer.pkl is corrupt (got {type(_loaded).__name__}) — re-fitting from train data")

if imp is None:
    imp = SimpleImputer(strategy="median").fit(X_tr)
    # Overwrite the bad file so future runs work correctly
    joblib.dump(imp, IMP_PATH)
    print(f"[3] Imputer re-fitted and saved to {IMP_PATH}")

X_tr_i = pd.DataFrame(imp.transform(X_tr), columns=FEATURE_COLS)
X_va_i = pd.DataFrame(imp.transform(X_va), columns=FEATURE_COLS)
X_te_i = pd.DataFrame(imp.transform(X_te), columns=FEATURE_COLS)

# ════════════════════════════════════════════
# 4. LOAD ALL PKL MODELS
# ════════════════════════════════════════════
PKL_FILES = {
    "XGBoost"         : "xgboost.pkl",
    "LightGBM"        : "lightgbm.pkl",
    "RandomForest"    : "randomforest.pkl",
    "GradientBoosting": "gradientboosting.pkl",
    "CatBoost"        : "catboost.pkl",
}

loaded_models = {}
for name, fname in PKL_FILES.items():
    path = MODEL_DIR / fname
    if path.exists():
        loaded_models[name] = joblib.load(path)
        print(f"[4] Loaded : {name:<20s} ← {fname}")
    else:
        print(f"[4] SKIP   : {name:<20s}   ({fname} not found)")

if not loaded_models:
    raise FileNotFoundError(f"No pkl models found in {MODEL_DIR}")

# ════════════════════════════════════════════
# 5. EVALUATE ON TRAIN / VAL / TEST
# ════════════════════════════════════════════
def compute_metrics(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)
    return {
        "R2"  : r2_score(y_true, y_pred),
        "MAE" : mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
    }

def predict(model, X, base=None):
    p = model.predict(X)
    if USE_RESID and base is not None:
        p = p + base
    return np.maximum(p, 0)

results = {}
for name, model in loaded_models.items():
    tr_pred = predict(model, X_tr_i, base_tr if USE_RESID else None)
    va_pred = predict(model, X_va_i, base_va if USE_RESID else None)
    te_pred = predict(model, X_te_i, base_te if USE_RESID else None)

    results[name] = {
        "train": compute_metrics(y_raw_tr, tr_pred),
        "val"  : compute_metrics(y_raw_va, va_pred),
        "test" : compute_metrics(y_raw_te, te_pred),
    }

# ════════════════════════════════════════════
# 6. PRINT RESULTS TABLE
# ════════════════════════════════════════════
print("\n" + "=" * 65)
print("  Per-Model Metrics Across Splits")
print("=" * 65)

HDR = f"  {'Model':<20s} {'Split':<7s} {'R²':>7s}  {'MAE':>8s}  {'RMSE':>8s}"
print(HDR)
print("  " + "-" * 61)

for name, splits in results.items():
    for split_name, m in splits.items():
        print(f"  {name if split_name=='train' else '':<20s} "
              f"{split_name:<7s} "
              f"{m['R2']:>7.4f}  "
              f"{m['MAE']:>8.2f}  "
              f"{m['RMSE']:>8.2f}")
    print("  " + "·" * 61)

# ════════════════════════════════════════════
# 7. OVERFIT GAP ANALYSIS
# ════════════════════════════════════════════
print("\n" + "=" * 65)
print("  Overfit Gap Analysis  (R²_train − R²_test)")
print("  Threshold: ≥ 0.10 → WARNING | ≥ 0.20 → OVERFIT")
print("=" * 65)
print(f"  {'Model':<20s} {'R²_train':>9s}  {'R²_val':>8s}  {'R²_test':>8s}  {'Gap(tr-te)':>11s}  Status")
print("  " + "-" * 65)

for name, splits in results.items():
    r2_tr = splits["train"]["R2"]
    r2_va = splits["val"]["R2"]
    r2_te = splits["test"]["R2"]
    gap   = r2_tr - r2_te

    if gap >= R2_GAP_ALERT:
        status = "🔴 OVERFIT"
    elif gap >= R2_GAP_WARN:
        status = "🟡 WARNING"
    else:
        status = "🟢 OK"

    print(f"  {name:<20s} {r2_tr:>9.4f}  {r2_va:>8.4f}  {r2_te:>8.4f}  {gap:>11.4f}  {status}")

# ════════════════════════════════════════════
# 8. MAE GAP ANALYSIS
# ════════════════════════════════════════════
print("\n" + "=" * 65)
print("  MAE Degradation  (MAE_test − MAE_train,  lower = better)")
print("=" * 65)
print(f"  {'Model':<20s} {'MAE_train':>10s}  {'MAE_val':>8s}  {'MAE_test':>9s}  {'Δ MAE':>7s}")
print("  " + "-" * 60)

for name, splits in results.items():
    mae_tr = splits["train"]["MAE"]
    mae_va = splits["val"]["MAE"]
    mae_te = splits["test"]["MAE"]
    delta  = mae_te - mae_tr
    print(f"  {name:<20s} {mae_tr:>10.2f}  {mae_va:>8.2f}  {mae_te:>9.2f}  {delta:>+7.2f}")

# ════════════════════════════════════════════
# 9. SUMMARY INTERPRETATION
# ════════════════════════════════════════════
print("\n" + "=" * 65)
print("  Interpretation Guide")
print("=" * 65)
print("""
  Gap (R²_train − R²_test)
  ─────────────────────────────────────────────────────────
  < 0.05   Perfect fit, model generalises well
  0.05–0.10  Slight overfit, acceptable for tree ensembles
  0.10–0.20  Moderate overfit → consider regularisation
             (↑ min_child_weight, ↑ min_samples_leaf,
              ↓ max_depth, ↓ n_estimators, ↑ reg_lambda)
  > 0.20   Severe overfit → retune hyperparameters or
             add more training data

  Train ≈ Val ≈ Test  → no overfit
  Train >> Val  → classical overfit (model memorised train)
  Val > Test   → val-set leakage or distribution shift
  Low R² everywhere → underfit or poor features
""")
print("=" * 65)

  Overfitting Diagnostic

[1] Loaded : 66,617 rows | 63 cols
[1] Features : 81
[2] Train 39,970 | Val 13,323 | Test 13,324
[3] imputer.pkl is corrupt (got float) — re-fitting from train data
[3] Imputer re-fitted and saved to C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\src\models\imputer.pkl
[4] Loaded : XGBoost              ← xgboost.pkl
[4] Loaded : LightGBM             ← lightgbm.pkl
[4] Loaded : RandomForest         ← randomforest.pkl
[4] Loaded : GradientBoosting     ← gradientboosting.pkl
[4] Loaded : CatBoost             ← catboost.pkl

  Per-Model Metrics Across Splits
  Model                Split        R²       MAE      RMSE
  -------------------------------------------------------------
  XGBoost              train    0.9625      5.89      8.46
                       val      0.9221      5.17      7.75
                       test     0.8117     11.77     19.25
  ·····························································
  LightGBM             train    0.9492    